# DS4DS Exercise Sheet 7

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

In [ ]:
using LinearAlgebra
using Plots
using StatsBase
using DelimitedFiles
using BenchmarkTools
using Optim

### Task 1: Non-linear function optimization from scratch 

In this task, we will implement our own non-linear optimization algorithm and apply it to the 2D Rosenbrock function. The function is defined by

$$
\begin{align}
    \mathcal{L}(\mathbf{w}) = (a - w_1)^2 + b (w_2 - w_1^2)^2
\end{align}
$$

where $a$ and $b$ are parameters of the function. In the following, the parameters will be set to $a = 1$ and $b = 100$.

#### a) Implement the Rosenbrock function so that the input is a vector $\mathbf{w}$ and it returns the scalar cost function value $\mathcal{L}(\mathbf{w})$. 

In [ ]:
function rosenbrock(w; a=1, b=100)
    ### BEGIN SOLUTION
    f_value = (a - w[1])^2 + b * (w[2] - w[1]^2)^2
    ### END SOLUTION
    return f_value
end

#### b) Derive the analytical gradient of the Rosenbrock function and implement it in the function outlined below so that it takes $\mathbf{w}$ as input and returns the gradient $\nabla\mathcal{L}(\mathbf{w})$ of the Rosenbrock function. 

In [ ]:
function rosenbrock_gradient(w; a=1, b=100)
    ### BEGIN SOLUTION
    gradient = [-2 * (a - w[1]) - 4 * b * w[1] * (w[2] - w[1]^2), 2 * b * (w[2] - w[1]^2)]
    ### END SOLUTION
    return gradient
end

#### c) Implement the steepest gradient descent rule for updating a parameter vector $\mathbf{w}_{current}$ to $\mathbf{w}_{next}$ in the function provided below. The step size $\eta$ is a parameter of this function. 

In [ ]:
function steepest_GD_parameter_update(gradient_function, w_current, eta)
    ### BEGIN SOLUTION
    w_next = w_current - eta * gradient_function(w_current)
    ### END SOLUTION
    return w_next
end

#### d) Implement the entire steepest descent method using the functions from the previous tasks using the template outlined below.

In [ ]:
function steepest_GD(f, grad_f, w_init, eta, max_iters=1000, tolerance=1e-5)
    """Steepest gradient descent implementation.

    Args:
        f: The function to be optimized
        grad_f: The analytical gradient of the function to be optimized
        w_init: The initial condition at which the optimization algorithm starts
        eta: The learning rate (here, the value is to be a constant during optimization)
        max_iters: A cap for the count of iterations
        tolerance: the tolerance value when the algorithm is terminated because the norm of the difference between
            the old and new point is too small (note that both max_iters and the tolerance cap can stop the descent,
            i.e., you need to check for both conditions)

    Returns:
        w_opt: The end value of the optimization

    """

    ### BEGIN SOLUTION

    w_current = w_init

    values = []
    for i in 1:max_iters
        w_next = steepest_GD_parameter_update(grad_f, w_current, eta)
        push!(values, f(w_next))
        if norm(w_next - w_current) < tolerance
            break
        end
        w_current = w_next
    end

    w_opt = w_current

    ### END SOLUTION

    return w_opt

end

#### e) Add a backtracking mechanism for selecting a step size that satisfies the Armijo condition. Hence, start with eta_init = 1 and then reduce it by a factor of 2 until the Armijo condition is satisfied. 

In [ ]:
function gradient_descent_backtracking(f, grad_f, w_init; eta_init=1, max_iters=1000, tolerance=1e-5)

    ### BEGIN SOLUTION

    w_current = w_init

    values = []
    for i in 1:max_iters
        grad = grad_f(w_current)
        eta = backtracking_line_search(f, grad_f, w_current, eta_init)
        w_next = w_current - eta * grad
        push!(values, f(w_next))
        if norm(w_next - w_current) < tolerance
            break
        end
        w_current = w_next
    end
    w_opt = w_current

    ### END SOLUTION
    return w_opt
end

function backtracking_line_search(f, grad_f, w_current, eta_init, alpha=0.3, beta=0.5)
    ### BEGIN SOLUTION
    eta = eta_init
    while f([w_current[1] - eta * grad_f(w_current)[1], w_current[2] - eta * grad_f(w_current)[2]]) > f(w_current) - alpha * eta * norm(grad_f(w_current))^2
        eta *= beta
    end
    ### END SOLUTION
    return eta
end

### Task 2: Fitting a surrogate model to geodata 

In this task, we are given a dataset of the geodata of north-western Germany and some parts of adjacent European countries. 
The dataset (obtained from [here](https://gdz.bkg.bund.de/index.php/default/digitale-geodaten/geodaetische-basisdaten/quasigeoid-der-bundesrepublik-deutschland-quasigeoid.html)) 
consists of a large number of points with two positions (longitude and latitude; $\mathbf{z} \in \mathbb{R}^2$) and an associated height value ($y \in \mathbb{R}$).

<img src="./approximate_map_area.png" alt="height_map" width="400"/>  <img src="./surface_top.svg" alt="data_height_map" width="400"/> <img src="./surface_front.svg" alt="data_height_map" width="400"/>

Source for topographic map (left-most image): https://de-de.topographic-map.com/map-95z57/Deutschland/

(There is some extrapolation issues at the sides of the plots, but these are not present in the data itself)

---

Your task is to fit a set of $q$ (slightly modified) radial basis functions (RBFs) to the data

$$
\begin{align}
    h(\mathbf{z}, \mathbf{w}) &= \sum_{i=1}^q \mathrm{RBF}(\mathbf{z}, \mathbf{w}_i) \\
    &=  \sum_{i=1}^q w_{i, 1} \cdot \exp\left( - \exp(w_{i, 2}) \cdot \left\| \begin{bmatrix} z_{1} \\ z_{2} \end{bmatrix} - \begin{bmatrix} w_{i, 3} \\ w_{i, 4} \end{bmatrix} \right\|_2^2\right),
\end{align}
$$

parameterized by $q$ parameter vectors $\mathbf{w}_i = \begin{bmatrix} w_{i, 1} \\ w_{i, 2} \\ w_{i, 3} \\ w_{i, 4} \end{bmatrix}$ for $i \in 1,\dots,q$ and the total (flattened) parameter vector $\mathbf{w}$ 

$$
\begin{align}
    \mathbf{w} = \begin{bmatrix} w_{1, 1} \\ w_{1, 2} \\  w_{1, 3} \\ \vdots \\ w_{q, 2} \\ w_{q, 3} \\ w_{q, 4} \\ \end{bmatrix}.
\end{align}
$$

##### **a)** Implement a single $\mathrm{RBF}(\mathbf{z}, \mathbf{w}_i)$ that takes $\mathbf{z}$ and the parameter vector $\mathbf{w}_i$ with four elements as input.

In [ ]:
function RBF(z, w_i)

    ### BEGIN SOLUTION

    f_value = w_i[1] * exp(-exp(w_i[2]) * norm(z - w_i[3:4])^2)

    ### END SOLUTION

    return f_value
end

##### **b)** Create a model $h(\mathbf{z}, \mathbf{w})$ that consists of a sum of $q$ RBF functions parameterized by a vector $\mathbf{w}$ with $4 \cdot q$ elements.

In [ ]:
function h(z, w, q)

    @assert length(w) == q * 4 "Invalid number of parameters."

    ### BEGIN SOLUTION

    f_value = 0
    for i in 1:q
        f_value += RBF(z, w[1+(i-1)*4:i*4])
    end

    ### END SOLUTION

    return f_value
end

##### **c)** Provide a function for the symbolic gradient of the RBF w.r.t. the weights, i.e., $\nabla_{\mathbf{w}} \mathrm{RBF}(\mathbf{z}, \mathbf{w})$.

In [ ]:
function gradient_RBF(z, w_i)

    ### BEGIN SOLUTION

    RBF_value = RBF(z, w_i)

    dfdwi1 = RBF_value / w_i[1]
    dfdwi2 = RBF_value * (-exp(w_i[2]) * norm(z - w_i[3:4])^2)
    dfdwi3 = RBF_value * (2 * exp(w_i[2]) * (z[1] - w_i[3]))
    dfdwi4 = RBF_value * (2 * exp(w_i[2]) * (z[2] - w_i[4]))

    gradient = [dfdwi1, dfdwi2, dfdwi3, dfdwi4]

    ### END SOLUTION

    return gradient
end

##### **d)** Provide a function for the gradient of the sum of $q$ RBF functions 
w.r.t. the weight vector $\mathbf{w}$, i.e., $\nabla_{\mathbf{w}} h(\mathbf{z}, \mathbf{w})$.

In [ ]:
function gradient_h(z, w, q)

    ### BEGIN SOLUTION

    gradient = zeros(q * 4)
    for j in 1:q
        gradient[1+(j-1)*4:j*4] = gradient_RBF(z, w[1+(j-1)*4:j*4])
    end

    ### END SOLUTION

    return gradient
end

##### **e)** In order to assess the model quality, we want to compare the mean squared error (MSE) between the data $y[k]$ and the prediction made by the trained model $y_{est}[k] = h(\mathbf{z}[k], \mathbf{w})$. Implement this function into the template below. $\mathbf{y}$ is the vector containing all elements of $y[k]$, i.e.,

$$
\begin{align}
    \mathbf{y} = \begin{bmatrix} y[1] \\ y[2] \\ \vdots \\ y[N] \end{bmatrix}
\end{align}
$$


and $\mathbf{Z}$ is a Matrix containing all $\mathbf{z}[k]$, i.e.,

$$
\begin{align}
    \mathbf{Z} = \begin{bmatrix} \mathbf{z}[1] \\ \mathbf{z}[2] \\ \vdots \\ \mathbf{z}[N] \end{bmatrix}
\end{align}.
$$

In [ ]:
function MSE_loss(y, Z, w, q)

    ### BEGIN SOLUTION

    N = size(Z, 1)
    y_est = [h(Z[k, :], w, q) for k in 1:N]
    loss_value = 1 / N * sum((y - y_est) .^ 2)

    return loss_value

    ### END SOLUTION

end

##### **f)** Provide a function that determines the gradient of the MSE loss w.r.t. the parameter vector $\mathbf{w}$, i.e., $\nabla_{\mathbf{w}} \mathrm{MSE}(\mathbf{y}, \mathbf{Z}, \mathbf{w})$. You may reuse the functions you introduced in previous subtasks.

In [ ]:
function gradient_MSE_loss(y, Z, w, q)

    ### BEGIN SOLUTION

    N = size(Z, 1)

    gradient = zeros(q * 4)

    for k in 1:N
        z_k = Z[k, :]
        y_k = y[k]

        model_gradient = gradient_h(z_k, w, q)
        prediction_error = y_k .- h(z_k, w, q)

        gradient += 2 * prediction_error .* (-model_gradient)
    end

    gradient = gradient / N

    ### END SOLUTION

    return gradient
end

##### Loading and preparing the dataset

In [ ]:
function load_dataset()
    ### load and prepare the data set
    X = readdlm("./GCG2016_WE.txt", Float64)
    N = size(X, 1)

    # find valid data points and store them in the 
    # arrays for position (Z) and height (y)
    Z = zeros(N, 2)
    y = zeros(N)
    s = 0
    for i in 1:N
        if X[i, 3] < 1e3
            s += 1
            Z[s, :] = X[i, 1:2]
            y[s] = X[i, 3]
        end
    end
    Z = Z[1:50:s, :]
    y = y[1:50:s]
    N = ceil(s / 50)

    print("Size of the final data set: ")
    println(N)

    # display(Plots.surface(Z[:, 2], Z[:, 1], y, camera=(135, 60), size=(600,600)))
    # savefig("surface_front.svg")

    # display(Plots.surface(Z[:, 2], Z[:, 1], y, camera=(0, 90), size=(600,600)))
    # savefig("surface_top.svg")

    function min_max_normalization(a)
        max = maximum(a)
        min = minimum(a)

        return (a .- min) ./ (max - min) .* 2 .- 1
    end

    y = min_max_normalization(y)

    Z_1 = min_max_normalization(Z[:, 1])
    Z_2 = min_max_normalization(Z[:, 2])

    Z = hcat(Z_1, Z_2)
    return Z, y, N
end

Z, y, N = load_dataset();

###

Code for the plotting of the data and your results is provided in the following cell. 

In [ ]:
display(Plots.plot(log.(loss_values), xlabel="iterations", ylabel="MSE", legend=false))

y_init = [h(Z[k, :], w0, q) for k in 1:size(Z, 1)]
display(Plots.surface(Z[:, 2], Z[:, 1], y_init, camera=(135, 60), title="initial guess"))

display(Plots.surface(Z[:, 2], Z[:, 1], y, camera=(135, 60), title="data set"))

y_est = [h(Z[k, :], w_opt, q) for k in 1:size(Z, 1)]
display(Plots.surface(Z[:, 2], Z[:, 1], y_est, camera=(135, 60), title="result"))

##### **g)** Use the standard gradient descent algorithm to fit a model with $q=100$ RBFs to the entire data set (use your own implementation, do not use the ```Optim```-package in this task). Run for $200$ iterations (without early termination) with step-length $\eta=3$ and use the given initial conditions $\mathbf{w}_0$ to initialize your algorithm.

Report on the MSE between the RBF model and the data set in each iteration of the algorithm (exactly $201$ loss values where the first one is from the initial guess and the last one is from after the last iteration). 

In [ ]:
function compute_initial_position(q)
    x_ = range(-1, stop=1, length=Integer(sqrt(q)))
    y_ = range(-1, stop=1, length=Integer(sqrt(q)))
    X = repeat(x_, Integer(sqrt(q)))[:]
    Y = repeat(y_', Integer(sqrt(q)))[:]
    gridPoints = [X Y]'

    w0 = [ones(q) * 0.05 ones(q) * 3.4 gridPoints']
    w0 = vcat(w0'...)
    return w0
end

In [ ]:
q = 100
w0 = compute_initial_position(q)
Z, y, N = load_dataset();
max_n_iterations = 200
eta = 3
loss_values = zeros(max_n_iterations + 1)  # fill this with your MSE loss values

### BEGIN SOLUTION

function fit_model_GD(
    y,
    Z,
    q,
    loss_function,
    gradient_loss_function,
    w0,
    max_n_iterations,
    eta
)

    loss_values = zeros(max_n_iterations + 1)
    loss_values[1] = loss_function(y, Z, w0, q)

    w_current = w0

    for i in 1:(max_n_iterations)

        if (i % 5 == 0)
            println(" current iteration: ", i, "\n current loss: ", loss_values[i], "\n")
        end

        g = gradient_loss_function(y, Z, w_current, q)
        w_current = w_current - eta * g
        loss_values[i+1] = MSE_loss(y, Z, w_current, q)
    end

    return w_current, loss_values

end


w_opt, loss_values = fit_model_GD(y, Z, q, MSE_loss, gradient_MSE_loss, w0, max_n_iterations, eta);
final_loss = loss_values[end];

println("final loss: ", final_loss)

### END SOLUTION

##### Food for thought: 
What is the influence of the parameter $w_{i, 2}$, why would one use $\exp(w_{i, 2})$ instead of simply $w_{i, 2}$ in the RBF? 